# 第 1 章 — Sella 入門と局所最小化

**ゴール**
- Sella を ASE の `Optimizer` 互換オブジェクトとして使えるようになる
- `order=0` (最小化) と `order=1` (鞍点) のスイッチを理解する
- 出力 trajectory を読み直してエネルギーの推移を可視化する

**題材**: Cu4 (テトラへドラル) クラスター を EMT (Effective Medium Theory) で最小化します。  
EMT は ASE 標準で追加インストール不要、Cu のような金属系を安価に扱えるので入門にぴったりです。

> EMT は H, C, N, O などの主要な軽元素は扱えません。気相の **有機分子** は 2 章 (xTB) から登場します。

In [ ]:
import numpy as np
from ase import Atoms
from ase.calculators.emt import EMT

# 1辺 ~2.5 Å の正四面体に少し歪みを加えた Cu4 を組む
a = 2.5
positions = np.array([
    [0.0, 0.0, 0.0],
    [a,   0.0, 0.0],
    [a/2, a*np.sqrt(3)/2, 0.0],
    [a/2, a*np.sqrt(3)/6, a*np.sqrt(6)/3],
])
rng = np.random.default_rng(0)
positions += rng.normal(scale=0.15, size=positions.shape)  # 0.15 Å の摂動

cluster = Atoms('Cu4', positions=positions)
cluster.center(vacuum=5.0)
cluster.calc = EMT()

print(f'初期エネルギー: {cluster.get_potential_energy():.4f} eV')
print(f'初期 fmax    : {np.linalg.norm(cluster.get_forces(), axis=1).max():.4f} eV/Å')

## Sella で最小化する

`Sella` クラスは ASE の `BFGS` などと同じ感覚で使えます。最小化したいときは `order=0` を渡します。

In [ ]:
from sella import Sella

opt = Sella(
    cluster,
    order=0,                       # 0: 最小化, 1: 1次鞍点 (TS)
    trajectory='cu4_min.traj',     # 各ステップを保存
    logfile='cu4_min.log',
)
converged = opt.run(fmax=1e-3, steps=200)
print(f'収束: {converged}')
print(f'最終エネルギー: {cluster.get_potential_energy():.4f} eV')

## Trajectory を読み直してプロット

`Trajectory` ファイルには各ステップの Atoms オブジェクトが入っています。
エネルギーと最大力 (fmax) の推移を可視化してみましょう。

In [ ]:
import matplotlib.pyplot as plt
from ase.io import Trajectory

traj = Trajectory('cu4_min.traj')
energies = [a.get_potential_energy() for a in traj]
fmaxes   = [np.linalg.norm(a.get_forces(), axis=1).max() for a in traj]

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(energies, marker='o')
ax[0].set_xlabel('step'); ax[0].set_ylabel('E [eV]'); ax[0].set_title('Energy')
ax[1].semilogy(fmaxes, marker='o')
ax[1].set_xlabel('step'); ax[1].set_ylabel('fmax [eV/Å]'); ax[1].set_title('Max force')
plt.tight_layout(); plt.show()

## BFGS と比較してみる

ASE の `BFGS` と Sella で、同じ初期構造から何ステップで収束するかを比較します。  
(EMT の Cu4 では大差は出ませんが、感覚を掴む練習として)

In [ ]:
from ase.optimize import BFGS

def fresh_cluster():
    a = Atoms('Cu4', positions=positions.copy())
    a.center(vacuum=5.0)
    a.calc = EMT()
    return a

# BFGS
atoms_bfgs = fresh_cluster()
bfgs = BFGS(atoms_bfgs, logfile=None)
bfgs.run(fmax=1e-3, steps=200)
n_bfgs = bfgs.nsteps

# Sella (order=0)
atoms_sella = fresh_cluster()
sella = Sella(atoms_sella, order=0, logfile=None)
sella.run(fmax=1e-3, steps=200)
n_sella = sella.nsteps

print(f'BFGS  : {n_bfgs} steps,  E = {atoms_bfgs.get_potential_energy():.5f} eV')
print(f'Sella : {n_sella} steps,  E = {atoms_sella.get_potential_energy():.5f} eV')

## 演習

1. 摂動の大きさ (`scale=0.15`) を `0.5` まで上げて、Sella と BFGS のどちらが安定して収束するかを比較してください。
2. `order=1` にして同じ Cu4 で実行すると何が起きるか観察してみましょう (鞍点には収束しないはずです — なぜ?)。
3. `cluster.center(vacuum=5.0)` を消すとどうなりますか? ASE が周期境界を仮定するときの注意点を確認してください。

---
次章では xTB を使って **HCN ⇌ HNC** の遷移状態を Sella で捕まえます。